In [1]:
%pip install datasets langchain langchain-community rank_bm25

   ---------------------------------------- 0.0/2.5 MB ? eta -:--:--
   - -------------------------------------- 0.1/2.5 MB 5.1 MB/s eta 0:00:01
   -------- ------------------------------- 0.6/2.5 MB 8.6 MB/s eta 0:00:01
   --------------------------- ------------ 1.8/2.5 MB 13.9 MB/s eta 0:00:01
   ---------------------------------------  2.5/2.5 MB 16.1 MB/s eta 0:00:01
   ---------------------------------------- 2.5/2.5 MB 14.7 MB/s eta 0:00:00
   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   -------------------------------------- - 1.0/1.0 MB 32.0 MB/s eta 0:00:01
   ---------------------------------------- 1.0/1.0 MB 21.9 MB/s eta 0:00:00
   ---------------------------------------- 0.0/61.0 kB ? eta -:--:--
   ---------------------------------------- 61.0/61.0 kB 3.2 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import datasets
from langchain_core.documents import Document

In [3]:
guest_dataset = datasets.load_dataset(
    "agents-course/unit3-invitees",
    split="train"
)

print(guest_dataset)

README.md:   0%|          | 0.00/371 [00:00<?, ?B/s]

c:\Users\maria\OneDrive\Documentos\HF_Agents_Course\venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\maria\.cache\huggingface\hub\datasets--agents-course--unit3-invitees. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


data/train-00000-of-00001.parquet:   0%|          | 0.00/3.32k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3 [00:00<?, ? examples/s]

Dataset({
    features: ['name', 'relation', 'description', 'email'],
    num_rows: 3
})


In [4]:
docs = [
    Document(
        page_content="\n".join([
            f"Name: {guest['name']}",
            f"Relation: {guest['relation']}",
            f"Description: {guest['description']}",
            f"Email: {guest['email']}"
        ]),
        metadata={"name": guest["name"]}
    )
    for guest in guest_dataset
]

In [5]:
print(docs[0])

page_content='Name: Ada Lovelace
Relation: best friend
Description: Lady Ada Lovelace is my best friend. She is an esteemed mathematician and friend. She is renowned for her pioneering work in mathematics and computing, often celebrated as the first computer programmer due to her work on Charles Babbage's Analytical Engine.
Email: ada.lovelace@example.com' metadata={'name': 'Ada Lovelace'}


In [6]:
from langchain_community.retrievers import BM25Retriever

In [7]:
bm25_retriever = BM25Retriever.from_documents(docs)

In [8]:
results = bm25_retriever.invoke("Ada Lovelace")

In [9]:
print(results[0].page_content)

Name: Ada Lovelace
Relation: best friend
Description: Lady Ada Lovelace is my best friend. She is an esteemed mathematician and friend. She is renowned for her pioneering work in mathematics and computing, often celebrated as the first computer programmer due to her work on Charles Babbage's Analytical Engine.
Email: ada.lovelace@example.com


In [10]:
from langchain_core.tools import Tool

In [11]:
def extract_text(query: str) -> str:
    
    results = bm25_retriever.invoke(query)
    
    if results:
        return "\n\n".join([
            doc.page_content for doc in results[:3]
        ])
    
    return "No matching guest information found."

In [12]:
guest_info_tool = Tool(
    name="guest_info_retriever",
    func=extract_text,
    description="Retrieves information about gala guests."
)

In [13]:
print(
    guest_info_tool.invoke("Nikola Tesla")
)

Name: Dr. Nikola Tesla
Relation: old friend from university days
Description: Dr. Nikola Tesla is an old friend from your university days. He's recently patented a new wireless energy transmission system and would be delighted to discuss it with you. Just remember he's passionate about pigeons, so that might make for good small talk.
Email: nikola.tesla@gmail.com

Name: Marie Curie
Relation: no relation
Description: Marie Curie was a groundbreaking physicist and chemist, famous for her research on radioactivity.
Email: marie.curie@example.com

Name: Ada Lovelace
Relation: best friend
Description: Lady Ada Lovelace is my best friend. She is an esteemed mathematician and friend. She is renowned for her pioneering work in mathematics and computing, often celebrated as the first computer programmer due to her work on Charles Babbage's Analytical Engine.
Email: ada.lovelace@example.com
